In [14]:
#Libraries used
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
import gc
import torch
from collections import defaultdict
import os
import scipy.io.wavfile as wav
import moviepy
from moviepy import *
from collections import defaultdict, Counter
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import shutil
import random
from sklearn.model_selection import train_test_split
from numpy.lib import stride_tricks

In [15]:
#Clear memory
gc.collect()
torch.cuda.empty_cache()

In [ ]:
#Using ESC+O ---> collapses output

In [9]:
#Training used for dog detection

In [ ]:
#Obejct Detection Dataset count
#dataset_path = "Dog_object_detection_dataset"
#splits = ["train", "val", "test"]
#data_counts = {}
#for split in splits:
#    image_folder = os.path.join(dataset_path, split, "images")
#    label_folder = os.path.join(dataset_path, split, "labels")
#    num_images = len([f for f in os.listdir(image_folder) if f.endswith(('.jpg', '.png', '.jpeg'))]) if os.path.exists(image_folder) else 0
#    num_labels = len([f for f in os.listdir(label_folder) if f.endswith('.txt')]) if os.path.exists(label_folder) else 0
#    data_counts[split] = {"images": num_images, "labels": num_labels}
#for split, counts in data_counts.items():
#    print(f"{split.capitalize()}:")
#    print(f"  Number of images: {counts['images']}")
#    print(f"  Number of labels: {counts['labels']}\n")

Train:
  Number of images: 6156
  Number of labels: 5168

Val:
  Number of images: 860
  Number of labels: 860

Test:
  Number of images: 892
  Number of labels: 892



In [ ]:
#Training of Dog Detection Model
#model = YOLO("yolov8n.pt") 
#results = model.train(data=r"dog_object_detection_dataset\data.yaml",epochs=10,imgsz=640,device="cuda",augment=True,)

In [ ]:
#List of models used in system 
image_model = YOLO(r'runs\classify\train3\weights\best.pt')
audio_model = YOLO(r'runs\classify\train12\weights\best.pt')
dog_detection_model=YOLO(r'runs\detect\train16\weights\best.pt')

In [ ]:
video_path = r'C:\Users\sujay\Desktop\project_2\videos\video.mp4' 

In [ ]:
def fourier_transformation(sig, frameSize, overlapFac=0.5, window=np.hanning): 
    win = window(frameSize)
    hopSize = int(frameSize - np.floor(overlapFac * frameSize))

    samples = np.append(np.zeros(int(np.floor(frameSize / 2.0))), sig)
    cols = np.ceil((len(samples) - frameSize) / float(hopSize)) + 1
    samples = np.append(samples, np.zeros(frameSize))

    frames = np.lib.stride_tricks.as_strided(
        samples, shape=(int(cols), frameSize),
        strides=(samples.strides[0] * hopSize, samples.strides[0])
    ).copy()

    frames *= win
    return np.fft.rfft(frames)

def make_logscale(spec, sr=44100, factor=20.):
    timebins, freqbins = np.shape(spec)
    scale = np.linspace(0, 1, freqbins) ** factor
    scale *= (freqbins - 1) / max(scale)
    scale = np.unique(np.round(scale))

    newspec = np.complex128(np.zeros([timebins, len(scale)]))
    for i in range(len(scale)):
        if i == len(scale) - 1:
            newspec[:, i] = np.sum(spec[:, int(scale[i]):], axis=1)
        else:
            newspec[:, i] = np.sum(spec[:, int(scale[i]):int(scale[i + 1])], axis=1)

    allfreqs = np.abs(np.fft.fftfreq(freqbins * 2, 1. / sr)[:freqbins + 1])
    freqs = [np.mean(allfreqs[int(scale[i]):int(scale[i + 1])]) if i < len(scale) - 1 else np.mean(allfreqs[int(scale[i]):]) for i in range(len(scale))]

    return newspec, freqs

def plot_spectrogram(location, categorie, output_folder, binsize=2**10, colormap="jet"):
    try:
        samplerate, samples = wav.read(location)
    except FileNotFoundError:
        print(f"Error: File not found -> {location}")
        return
    
    s = fourier_transformation(samples, binsize)
    sshow, freq = make_logscale(s, factor=1.0, sr=samplerate)

    ims = 20. * np.log10(np.abs(sshow) / 10e-6)

    timebins, freqbins = np.shape(ims)
    print(f"Processing {categorie}: {location}")
    
    plt.figure(figsize=(15, 7.5))
    plt.title(f'Class Label: {categorie}')
    plt.imshow(np.transpose(ims), origin="lower", aspect="auto", cmap=colormap, interpolation="none")
    plt.colorbar()
    
    plt.xlabel("time (s)")
    plt.ylabel("frequency (hz)")
    plt.xlim([0, timebins - 1])
    plt.ylim([0, freqbins])
    os.makedirs(output_folder, exist_ok=True)
    filename = os.path.basename(location).replace('.wav', '.png')
    save_path = os.path.join(output_folder, filename)
    plt.savefig(save_path, bbox_inches="tight")
    plt.close()
    print(f"Saved spectrogram: {save_path}")

def test_vis(location, filepath, binsize=2**10, colormap="jet"):
    samplerate, samples = wav.read(location)
    s = fourier_transformation(samples, binsize)
    sshow, freq = make_logscale(s, factor=1.0, sr=samplerate)
    with np.errstate(divide='ignore'):
        ims = 20.*np.log10(np.abs(sshow)/10e-6)
    timebins, freqbins = np.shape(ims)
    plt.figure(figsize=(15, 7.5))
    plt.imshow(np.transpose(ims), origin="lower", aspect="auto", cmap=colormap, interpolation="none")
    plt.axis('off')
    plt.xlim([0, timebins-1])
    plt.ylim([0, freqbins])
    plt.savefig(filepath, bbox_inches="tight")
    plt.close()
    return

In [ ]:
detection_model = dog_detection_model 10
emotion_classification_model = image_model
audio_classification_model=audio_model
dog_emotions = defaultdict(list)
emotion_labels = {0: "angry", 1: "happy", 2: "relaxed", 3: "sad"}
def detect_dogs(frame):
    results = detection_model(frame)
    detections = []
    for result in results:
        for box in result.boxes.data:
            x1, y1, x2, y2, conf = box[:5].tolist()
            detections.append(((int(x1), int(y1), int(x2 - x1), int(y2 - y1)), conf))
    return detections
def classify_emotion(cropped_dog):
    cropped_dog = cv2.resize(cropped_dog, (256, 256))
    cropped_dog = cv2.cvtColor(cropped_dog, cv2.COLOR_BGR2GRAY)
    cropped_dog = np.expand_dims(cropped_dog, axis=-1)
    results = emotion_classification_model(cropped_dog)
    if results and results[0].probs is not None:
        probs = results[0].probs.data.cpu().numpy()
        top_emotion = results[0].probs.top1
        return top_emotion, probs
    return None, None
def iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    interArea = max(0, xB - xA) * max(0, yB - yA)
    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    iou_score = interArea / float(boxAArea + boxBArea - interArea)
    return iou_score
def process_video(video_path):
    cap = cv2.VideoCapture(video_path)
    unique_dogs = {}
    frame_idx = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        detections = detect_dogs(frame)
        if not detections:
            continue
        new_unique_dogs = {}
        for bbox, conf in detections:
            x, y, w, h = bbox
            cropped_dog = frame[y:y+h, x:x+w]

            emotion, prob = classify_emotion(cropped_dog)
            if emotion is None:
                continue 
            matched_id = None
            for dog_id, prev_bbox in unique_dogs.items():
                if iou(prev_bbox, (x, y, x + w, y + h)) > 0.5:  
                    matched_id = dog_id
                    break
            
            if matched_id is None:
                matched_id = len(unique_dogs) + 1  
            new_unique_dogs[matched_id] = (x, y, x + w, y + h)
            dog_emotions[matched_id].append(prob) 
        unique_dogs = new_unique_dogs
        frame_idx += 1
    cap.release()
    final_emotions = {}
    for dog_id, probs in dog_emotions.items():
        avg_probs = np.mean(probs, axis=0)
        most_probable_emotion = np.argmax(avg_probs)
        final_emotions[dog_id] = emotion_labels[most_probable_emotion]
    return final_emotions
dog_emotion_results = process_video(video_path)
for dog_id, emotion in dog_emotion_results.items():
    print(f"Dog {dog_id}: Most Probable Emotion = {emotion}")
clip = moviepy.VideoFileClip(video_path)
clip.audio.write_audiofile(r"audio_extracted_from_video\output_audio.wav")
output_audio_path=r"audio_extracted_from_video\output_audio.wav"
filename = os.path.splitext(os.path.basename(output_audio_path))[0]
test_vis(output_audio_path,filepath=f'audio_extracted_from_video_spectrogram\{filename}.jpg')
pred = audio_classification_model(f'audio_extracted_from_video_spectrogram\{filename}.jpg')


0: 640x384 1 dog, 85.9ms
Speed: 1.9ms preprocess, 85.9ms inference, 11.0ms postprocess per image at shape (1, 3, 640, 384)

0: 256x256 relaxed 1.00, happy 0.00, angry 0.00, sad 0.00, 4.5ms
Speed: 13.2ms preprocess, 4.5ms inference, 0.0ms postprocess per image at shape (1, 3, 256, 256)

0: 640x384 1 dog, 19.2ms
Speed: 1.3ms preprocess, 19.2ms inference, 1.8ms postprocess per image at shape (1, 3, 640, 384)

0: 256x256 relaxed 1.00, happy 0.00, sad 0.00, angry 0.00, 5.0ms
Speed: 2.0ms preprocess, 5.0ms inference, 0.1ms postprocess per image at shape (1, 3, 256, 256)

0: 640x384 1 dog, 10.9ms
Speed: 1.3ms preprocess, 10.9ms inference, 1.8ms postprocess per image at shape (1, 3, 640, 384)

0: 256x256 relaxed 1.00, happy 0.00, sad 0.00, angry 0.00, 6.3ms
Speed: 1.8ms preprocess, 6.3ms inference, 0.0ms postprocess per image at shape (1, 3, 256, 256)

0: 640x384 1 dog, 11.0ms
Speed: 1.2ms preprocess, 11.0ms inference, 2.1ms postprocess per image at shape (1, 3, 640, 384)

0: 256x256 relaxed 

MoviePy - Done.



image 1/1 c:\Users\sujay\Desktop\project_2\audio_extracted_from_video_spectrogram\output_audio.jpg: 640x640 growl 0.91, grunt 0.09, bark 0.00, 5.3ms
Speed: 20.8ms preprocess, 5.3ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)
